## Runtime Setup

This notebook supports both local execution and Google Colab.

Local:
1. Open VS Code from inside the `XAllergen2.0` project root.
2. Select the `Python (xallergen2)` kernel or the repo `.venv`.
3. Run the notebook from top to bottom.

Colab:
1. Run the setup cell first.
2. It mounts Google Drive.
3. It installs the notebook-specific Python packages.
4. It downloads `baseline_notebook_utils.py` plus the train/test CSVs from GitHub into the runtime.
5. It saves generated checkpoints and metrics back to Google Drive.

Expected outputs:
- Checkpoint: `models/baseline_frozen_esm2.pt`
- Metrics: `results/baseline_metrics.json`


# Frozen ESM-2 baseline for DeepAlgPro

This notebook trains a frozen `esm2_t6_8M_UR50D` allergenicity classifier on `data/deepalgpro_train_cleaned.csv` and evaluates it on the held-out `data/deepalgpro_test_cleaned.csv`.

Architecture:
- Backbone: `esm2_t6_8M_UR50D`, frozen throughout training.
- Attention pooling: `Linear(embed_dim -> 1)` per residue, softmax-normalized across sequence length, then a weighted sum of residue embeddings. The resulting per-residue weights are kept as an attribution signal.
- Classification head: `Linear(embed_dim -> 128) -> ReLU -> Dropout(0.3) -> Linear(128 -> 1)`.
- Training loss: `BCEWithLogitsLoss`; inference uses `sigmoid`.

Split strategy:
- `deepalgpro_train_cleaned.csv` is split into train and validation with a stratified random 90/10 split.
- `sklearn.model_selection.train_test_split` is used with `stratify=df["label"]` and `random_state=42`.
- `deepalgpro_test_cleaned.csv` remains fully held out until final evaluation.

In [ ]:
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
RUN_TARGET = 'colab' if IN_COLAB else 'local'
os.environ['XALLERGEN_RUN_TARGET'] = RUN_TARGET

REPO_OWNER = 'Jeffateth'
REPO_NAME = 'XAllergen2.0'
REPO_BRANCH = 'main'
REPO_RAW_BASE = f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{REPO_BRANCH}'

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            'captum==0.8.0',
            'transformers==4.48.1',
            'sentencepiece==0.2.1',
            'seaborn==0.13.2',
        ],
        check=True,
    )

    PROJECT_ROOT = Path('/content') / REPO_NAME
    DATA_DIR = PROJECT_ROOT / 'data'
    MODEL_DIR = PROJECT_ROOT / 'models'
    RESULTS_DIR = PROJECT_ROOT / 'results'
    for folder in [PROJECT_ROOT, DATA_DIR, MODEL_DIR, RESULTS_DIR]:
        folder.mkdir(parents=True, exist_ok=True)

    downloads = {
        'baseline_notebook_utils.py': PROJECT_ROOT / 'baseline_notebook_utils.py',
        'data/deepalgpro_train_cleaned.csv': DATA_DIR / 'deepalgpro_train_cleaned.csv',
        'data/deepalgpro_test_cleaned.csv': DATA_DIR / 'deepalgpro_test_cleaned.csv',
    }
    for rel_path, destination in downloads.items():
        urllib.request.urlretrieve(f'{REPO_RAW_BASE}/{rel_path}', destination)

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)

    DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/XAllergen2.0') / 'baseline_colab'
    DRIVE_MODEL_DIR = DRIVE_RUN_ROOT / 'models'
    DRIVE_RESULTS_DIR = DRIVE_RUN_ROOT / 'results'
    for folder in [DRIVE_MODEL_DIR, DRIVE_RESULTS_DIR]:
        folder.mkdir(parents=True, exist_ok=True)
else:
    PROJECT_ROOT = Path.cwd()
    DATA_DIR = PROJECT_ROOT / 'data'
    MODEL_DIR = PROJECT_ROOT / 'models'
    RESULTS_DIR = PROJECT_ROOT / 'results'
    DRIVE_RUN_ROOT = None
    DRIVE_MODEL_DIR = None
    DRIVE_RESULTS_DIR = None


def sync_outputs_to_drive(*paths: Path) -> list[Path]:
    if not IN_COLAB or DRIVE_RUN_ROOT is None:
        return []

    copied_paths = []
    for path in paths:
        src = Path(path)
        if not src.exists():
            continue
        if src.parent.name == 'models':
            dest = DRIVE_MODEL_DIR / src.name
        elif src.parent.name == 'results':
            dest = DRIVE_RESULTS_DIR / src.name
        else:
            dest = DRIVE_RUN_ROOT / src.name
        shutil.copy2(src, dest)
        copied_paths.append(dest)
    return copied_paths

print(f'RUN_TARGET bootstrap: {RUN_TARGET}')
print(f'PROJECT_ROOT bootstrap: {PROJECT_ROOT}')
if IN_COLAB:
    print(f'Drive output root: {DRIVE_RUN_ROOT}')


In [ ]:
import json
import math
import time as _time
from pathlib import Path

from baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    RANDOM_STATE,
    THRESHOLD,
    FrozenESMAllergenClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    normalize_scores,
    print_runtime_context,
    seed_everything,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
configure_matplotlib_cache(PROJECT_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
import sys
from tqdm import tqdm

BATCH_SIZE = 24
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV = DATA_DIR / 'deepalgpro_test_cleaned.csv'
CHECKPOINT_PATH = MODEL_DIR / 'baseline_frozen_esm2.pt'
METRICS_PATH = RESULTS_DIR / 'baseline_metrics.json'
DRIVE_CHECKPOINT_PATH = DRIVE_MODEL_DIR / CHECKPOINT_PATH.name if DRIVE_MODEL_DIR else None
DRIVE_METRICS_PATH = DRIVE_RESULTS_DIR / METRICS_PATH.name if DRIVE_RESULTS_DIR else None

seed_everything(RANDOM_STATE)
DEVICE = detect_device()
print_runtime_context(DEVICE, PROJECT_ROOT)


In [ ]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df["sequence_id"] = df["sequence_id"].astype(str)
    df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
    df["label"] = df["label"].astype(int)

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df["label"],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df = val_split_df.reset_index(drop=True).copy()

print(f"Train sequences: {len(train_split_df):,}")
print(train_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Validation sequences: {len(val_split_df):,}")
print(val_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Held-out test sequences: {len(test_df):,}")
print(test_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        return {
            "sequence_id": row["sequence_id"],
            "sequence": row["sequence"],
            "label": int(row["label"]),
        }


tokenizer = build_tokenizer(HF_MODEL_NAME)


def collate_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
    }


def move_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    moved["input_ids"] = batch["input_ids"].to(device)
    moved["attention_mask"] = batch["attention_mask"].to(device)
    moved["label"] = batch["label"].to(device)
    return moved


train_loader = DataLoader(SequenceDataset(train_split_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(SequenceDataset(val_split_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(SequenceDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)

model = FrozenESMAllergenClassifier(HF_MODEL_NAME, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
assert not any(param.requires_grad for param in model.backbone.parameters())
trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")


In [ ]:
from typing import Optional, Tuple
from tqdm.auto import tqdm

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
    epoch: int,
) -> float:
    model.train()
    total_loss = 0.0
    total_examples = 0
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss = criterion(outputs["logits"], batch["label"])
        loss.backward()
        optimizer.step()
        batch_size = batch["label"].shape[0]
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
    return total_loss / max(total_examples, 1)


@torch.no_grad()
def predict(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    criterion: Optional[nn.Module] = None,
    desc: Optional[str] = None,
) -> Tuple[Optional[float], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    rows = []
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        logits = outputs["logits"]
        probs = torch.sigmoid(logits)
        if criterion is not None:
            loss = criterion(logits, batch["label"])
            batch_size = batch["label"].shape[0]
            total_loss += float(loss.item()) * batch_size
            total_examples += batch_size
        for idx in range(len(batch["sequence_id"])):
            rows.append(
                {
                    "sequence_id": batch["sequence_id"][idx],
                    "sequence": batch["sequence"][idx],
                    "label": int(batch["label"][idx].item()),
                    "logit": float(logits[idx].item()),
                    "pred_prob": float(probs[idx].item()),
                    "pred_label": int(probs[idx].item() >= THRESHOLD),
                }
            )
    loss_value = None if criterion is None else total_loss / max(total_examples, 1)
    return loss_value, pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    return metrics


history = []
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

_epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch", file=sys.stdout, dynamic_ncols=True)
for epoch in _epoch_bar:
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    val_loss, val_pred_df = predict(
        model, val_loader, DEVICE, criterion,
    )
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                },
                "training_history": history,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    # ← print each epoch result as a plain line so you always see progress
    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"best={best_epoch}"
    )
    _epoch_bar.set_postfix(
        train_loss=f"{train_loss:.5f}",
        val_loss=f"{val_loss:.5f}",
        best=best_epoch,
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation loss: {best_val_loss:.5f} at epoch {best_epoch}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")

In [ ]:
model, checkpoint = load_baseline_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)

train_history_df = pd.DataFrame(checkpoint['training_history'])

val_loss, val_predictions_df = predict(model, val_loader, DEVICE, criterion)
test_loss, test_predictions_df = predict(model, test_loader, DEVICE, criterion)

test_metrics = compute_metrics(test_predictions_df)
test_metrics['test_loss'] = float(test_loss)
test_metrics['best_epoch'] = int(best_epoch)
test_metrics['n_test_sequences'] = int(len(test_predictions_df))

metrics_payload = {
    'esm_model_name': ESM_MODEL_NAME,
    'architecture_hyperparameters': {'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT},
    'training': {
        'batch_size': BATCH_SIZE,
        'epochs_requested': EPOCHS,
        'early_stopping_patience': PATIENCE,
        'optimizer': 'AdamW',
        'lr': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
    },
    'validation_loss': float(val_loss),
    'test_metrics': test_metrics,
}

with METRICS_PATH.open('w') as handle:
    json.dump(metrics_payload, handle, indent=2)

SYNCED_OUTPUTS = sync_outputs_to_drive(CHECKPOINT_PATH, METRICS_PATH)


In [ ]:
print(f'Run target: {RUN_TARGET}')
print('Runtime outputs saved to disk:')
print(f'  {CHECKPOINT_PATH}')
print(f'  {METRICS_PATH}')
if SYNCED_OUTPUTS:
    print('Copied outputs to Google Drive:')
    for synced_path in SYNCED_OUTPUTS:
        print(f'  {synced_path}')


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_history_df["epoch"], train_history_df["train_loss"], label="Train loss")
axes[0].plot(train_history_df["epoch"], train_history_df["val_loss"], label="Validation loss")
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

cm = confusion_matrix(test_predictions_df["label"], test_predictions_df["pred_label"], labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("Held-out test confusion matrix")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
axes[1].set_xticklabels(["Negative", "Positive"])
axes[1].set_yticklabels(["Negative", "Positive"], rotation=0)

plt.tight_layout()
plt.show()

display(Markdown("## Test metrics"))
display(pd.DataFrame([{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}]).T.rename(columns={0: 'value'}))
print(f"Saved metrics JSON to: {METRICS_PATH}")

In [ ]:
def select_attribution_examples(pred_df: pd.DataFrame) -> pd.DataFrame:
    selected_frames = []
    selected_ids = set()

    high_conf_tp = pred_df.loc[
        (pred_df["label"] == 1)
        & (pred_df["pred_label"] == 1)
        & (pred_df["pred_prob"] > 0.9)
    ]
    if len(high_conf_tp) > 0:
        tp_sample = high_conf_tp.sample(n=min(3, len(high_conf_tp)), random_state=RANDOM_STATE)
        selected_frames.append(tp_sample)
        selected_ids.update(tp_sample["sequence_id"])

    errors = pred_df.loc[pred_df["pred_label"] != pred_df["label"]]
    if len(errors) > 0:
        error_sample = errors.sample(n=min(2, len(errors)), random_state=RANDOM_STATE)
        selected_frames.append(error_sample)
        selected_ids.update(error_sample["sequence_id"])

    selected_df = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame(columns=pred_df.columns)

    if len(selected_df) < 5:
        remainder = pred_df.loc[~pred_df["sequence_id"].isin(selected_ids)]
        fill_n = min(5 - len(selected_df), len(remainder))
        if fill_n > 0:
            filler = remainder.sample(n=fill_n, random_state=RANDOM_STATE)
            selected_df = pd.concat([selected_df, filler], ignore_index=True)

    return selected_df.head(5).copy()


def compute_attention_and_ig(sequence: str) -> tuple[np.ndarray, np.ndarray]:
    attention_scores = normalize_scores(
        compute_attention_weights(model, tokenizer, sequence, DEVICE)
    )
    ig_scores = compute_integrated_gradients(
        model,
        tokenizer,
        sequence,
        DEVICE,
        steps=IG_STEPS,
        normalize=True,
    )
    return attention_scores, ig_scores


## Wrap-up

This notebook saves the frozen ESM-2 baseline checkpoint to `models/baseline_frozen_esm2.pt` and the held-out test metrics to `results/baseline_metrics.json`. Review the evaluation plots and test metrics above as the baseline reference point.

The next step is dataset-scale attribution evaluation against IEDB epitope ground truth in `03_attribution_evaluation.ipynb`. The attribution plots here are only a small visual sanity check and should not be interpreted as epitope-level validation.